In [ ]:
#!/usr/bin/env python
"""
Notebook: 06_hitl_simulation.ipynb
Human-in-the-Loop Simulation for ECG Anomaly Detection
"""

 # Human-in-the-Loop (HITL) Simulation
 
# This notebook demonstrates the HITL workflow:
 - Queue management for low-confidence predictions
 - Expert review simulation
 - Active learning sample selection
 - Model improvement from feedback

# 1. Setup and Imports

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import time

sys.path.insert(0, '..')

from src.hitl.review_queue import ReviewQueue, QueueConfig, ReviewPriority
from src.hitl.reviewer import Reviewer, ExpertFeedback
from src.hitl.active_learning import ActiveLearner, SamplingStrategy, ActiveLearningConfig
from src.hitl.feedback_store import FeedbackStore

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# 2. Initialize HITL Components

In [ ]:
# Initialize review queue
queue_config = QueueConfig(
    max_size=100,
    critical_timeout_seconds=300,
    high_timeout_seconds=3600,
    enable_escalation=True
)
review_queue = ReviewQueue(queue_config)

# Initialize feedback store
feedback_store = FeedbackStore(storage_path="./data/hitl_feedback")

# Initialize active learner
active_config = ActiveLearningConfig(
    strategy=SamplingStrategy.HYBRID,
    batch_size=20,
    uncertainty_threshold=0.3
)
active_learner = ActiveLearner(active_config)

# Initialize reviewer
reviewer = Reviewer(reviewer_id="expert_dr_smith", expertise_level="expert")

print("HITL components initialized successfully")

# 3. Simulate Model Predictions

In [ ]:
# Generate synthetic predictions
np.random.seed(42)
num_samples = 100

# Simulate model outputs
predictions = []
for i in range(num_samples):
    confidence = np.random.beta(2, 5)  # Skewed towards lower confidence
    class_probs = np.random.dirichlet([1, 1, 1, 1, 1])
    
    # Some samples have very low confidence (uncertain)
    if i > 80:  # Last 20 samples are more uncertain
        confidence = np.random.beta(1, 10)
        class_probs = np.array([0.25, 0.25, 0.25, 0.15, 0.10])
    
    pred_class = np.argmax(class_probs)
    class_names = ['Normal', 'AFib', 'PVC', 'Bradycardia', 'Other']
    
    predictions.append({
        'id': f"sample_{i:03d}",
        'confidence': confidence,
        'predicted_class': class_names[pred_class],
        'probabilities': class_probs.tolist(),
        'beat_data': np.random.randn(187)  # Synthetic beat
    })

print(f"Generated {len(predictions)} predictions")
print(f"Average confidence: {np.mean([p['confidence'] for p in predictions]):.3f}")

# 4. Queue Low-Confidence Predictions for Review

In [ ]:
# Add low-confidence predictions to review queue
items_added = 0
for pred in predictions:
    if pred['confidence'] < 0.7:  # Confidence threshold
        review_queue.add_item(
            beat_data=pred['beat_data'],
            model_prediction=pred,
            priority=None,  # Auto-determine priority
            metadata={'source': 'simulation'}
        )
        items_added += 1

print(f"Added {items_added} items to review queue")
print(f"Queue size: {review_queue._get_total_size()}")

# Check queue statistics
stats = review_queue.get_stats()
print("\nQueue Statistics:")
print(f"  Pending items: {stats['total_pending']}")

# 5. Simulate Expert Review Process

In [ ]:
# Start review session
session_id = reviewer.start_session()
print(f"Started review session: {session_id}")

# Review items from queue
reviewed_items = []
review_results = []

for _ in range(15):  # Review 15 items
    item = review_queue.get_next_item(reviewer_id=reviewer.reviewer_id)
    
    if item is None:
        break
        
    # Simulate expert review
    true_label = item.model_prediction['predicted_class']
    
    # Expert corrects with 90% accuracy
    if np.random.random() < 0.9:
        corrected_label = true_label
    else:
        # Random correction
        class_names = ['Normal', 'AFib', 'PVC', 'Bradycardia', 'Other']
        class_names.remove(true_label)
        corrected_label = np.random.choice(class_names)
    
    # Submit feedback
    feedback = reviewer.submit_feedback(
        item_id=item.id,
        corrected_label=corrected_label,
        confidence=0.95,
        notes="Simulated expert review",
        escalate=np.random.random() < 0.1  # 10% escalation rate
    )
    
    if feedback:
        # Mark queue item as completed
        review_queue.complete_review(item.id, {
            'corrected_label': corrected_label,
            'reviewer_id': reviewer.reviewer_id
        })
        
        # Store feedback
        feedback_store.add_feedback(feedback, item.model_prediction)
        
        reviewed_items.append({
            'item_id': item.id,
            'original': item.model_prediction['predicted_class'],
            'corrected': corrected_label,
            'was_corrected': original != corrected_label,
            'confidence': item.model_prediction['confidence']
        })

# End session
reviewer.end_session()

print(f"\nReviewed {len(reviewed_items)} items")
print(f"Correction rate: {sum(1 for r in reviewed_items if r['was_corrected']) / len(reviewed_items):.2%}")

# 6. Active Learning Sample Selection

In [ ]:
# Use active learning to select most informative samples
unlabeled_pool = predictions[:50]  # First 50 as unlabeled pool

selected_for_review = active_learner.select_samples_for_review(
    unlabeled_pool,
    num_samples=10
)

print(f"Selected {len(selected_for_review)} samples for review")
print("\nSelected sample confidences:")
for sample in selected_for_review:
    print(f"  {sample['id']}: confidence={sample['confidence']:.3f}, "
          f"prediction={sample['predicted_class']}")

# Visualize active learning strategy
fig, ax = plt.subplots(figsize=(10, 6))
confidences = [s['confidence'] for s in unlabeled_pool]
selected_confidences = [s['confidence'] for s in selected_for_review]

ax.hist(confidences, bins=20, alpha=0.5, label='Unlabeled Pool', edgecolor='black')
ax.hist(selected_confidences, bins=20, alpha=0.7, label='Selected for Review', edgecolor='black')
ax.axvline(active_learner.config.uncertainty_threshold, color='red', 
          linestyle='--', label='Uncertainty Threshold')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Frequency')
ax.set_title(f'Active Learning Sampling - Strategy: {active_learner.config.strategy.value}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 7. Track Model Improvement from Feedback

In [ ]:
# Simulate model retraining with feedback
def simulate_model_improvement(feedback_count):
    """Simulate model accuracy improvement from feedback"""
    base_accuracy = 0.85
    improvement = 0.15 * (1 - np.exp(-feedback_count / 100))
    return base_accuracy + improvement

feedback_counts = np.arange(0, 201, 20)
accuracies = [simulate_model_improvement(count) for count in feedback_counts]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(feedback_counts, accuracies, 'b-o', linewidth=2, markersize=8)
ax.axhline(0.85, color='red', linestyle='--', label='Baseline (No Feedback)')
ax.set_xlabel('Number of Feedback Samples')
ax.set_ylabel('Model Accuracy')
ax.set_title('Model Improvement from Expert Feedback')
ax.legend()
ax.grid(True, alpha=0.3)

# Add annotations
ax.annotate(f'+{(accuracies[-1] - 0.85)*100:.1f}% improvement',
           xy=(200, accuracies[-1]), xytext=(150, accuracies[-1] + 0.02),
           arrowprops=dict(arrowstyle='->', color='green'))

plt.tight_layout()
plt.show()

# 8. Feedback Analytics and Reporting

In [ ]:
# Generate feedback statistics
stats = feedback_store.get_stats()

print("=" * 60)
print("HITL System Statistics")
print("=" * 60)
print(f"Total feedback received: {stats.total_feedback}")
print(f"Used for training: {stats.used_for_training}")
print(f"Correction rate: {stats.correction_rate:.2%}")
print(f"Escalation rate: {stats.escalation_rate:.2%}")
print(f"Average reviewer confidence: {stats.avg_reviewer_confidence:.3f}")
print("\nFeedback by class:")
for class_name, count in sorted(stats.by_class.items()):
    print(f"  {class_name}: {count}")

# Analyze confusion between model and experts
confusion_analysis = feedback_store.get_confusion_analysis()
print("\nMisclassification Rate by Original Class:")
for class_name, rate in confusion_analysis.get('misclassification_rate', {}).items():
    print(f"  {class_name}: {rate:.2%}")

# 9. Visualization of HITL Workflow

In [ ]:
# Create HITL workflow diagram
fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

# Define workflow steps
steps = [
    (1, 7, "Model Inference"),
    (3, 7, "Confidence Score"),
    (5, 7, "Threshold Check"),
    (7, 7, "High Confidence"),
    (9, 7, "Auto-Classify"),
    (3, 3, "Low Confidence"),
    (5, 3, "Add to Queue"),
    (7, 3, "Expert Review"),
    (9, 3, "Correction & Feedback")
]

# Draw boxes
for x, y, label in steps:
    rect = plt.Rectangle((x-0.8, y-0.5), 1.6, 1, 
                         facecolor='lightblue', edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

# Draw arrows
arrows = [
    ((1.8, 7), (2.2, 7)),  # Model -> Confidence
    ((3.8, 7), (4.2, 7)),  # Confidence -> Threshold
    ((5.8, 7), (6.2, 7)),  # Threshold -> High Confidence
    ((7.8, 7), (8.2, 7)),  # High Confidence -> Auto-Classify
    ((5.8, 6.5), (5.8, 3.5)),  # Threshold -> Low Confidence (down)
    ((6.2, 3), (6.8, 3)),  # Low Confidence -> Queue
    ((7.8, 3), (8.2, 3)),  # Queue -> Expert
    ((8.2, 3.5), (8.2, 6.5)),  # Expert -> Feedback (up)
]

for start, end in arrows:
    ax.annotate('', xy=end, xytext=start,
               arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

ax.set_title('Human-in-the-Loop Workflow for ECG Anomaly Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 10. Simulate Real-time HITL Dashboard

In [ ]:
# Simulate real-time dashboard updates
def simulate_dashboard():
    """Simulate a real-time HITL dashboard"""
    print("\n" + "="*60)
    print("HITL DASHBOARD - REAL-TIME MONITORING")
    print("="*60)
    
    for i in range(10):
        time.sleep(0.5)  # Simulate real-time updates
        
        # Update metrics
        queue_size = review_queue._get_total_size()
        review_rate = len(feedback_store.feedback_records)
        correction_rate = stats.correction_rate if stats.total_feedback > 0 else 0
        
        print(f"\n[Update {i+1}] - {datetime.now().strftime('%H:%M:%S')}")
        print(f"  Queue Size: {queue_size} pending")
        print(f"  Total Reviews: {review_rate}")
        print(f"  Correction Rate: {correction_rate:.2%}")
        print(f"  Active Reviewer: {reviewer.reviewer_id}")
        
        # Progress bar for queue
        progress = 1 - (queue_size / queue_config.max_size)
        bar_length = 30
        filled = int(bar_length * progress)
        bar = '█' * filled + '░' * (bar_length - filled)
        print(f"  Queue Capacity: [{bar}] {queue_size}/{queue_config.max_size}")

simulate_dashboard()

# 11. Export HITL Data

In [ ]:
# Export feedback for model retraining
export_path = "./data/hitl_training_data.json"
feedback_store.export_for_training(export_path, include_features=False)

print(f"\nExported feedback data to {export_path}")

# Summary report
print("\n" + "="*60)
print("HITL SIMULATION SUMMARY")
print("="*60)
print(f"Total predictions processed: {len(predictions)}")
print(f"Low-confidence flagged: {items_added}")
print(f"Expert reviews completed: {len(reviewed_items)}")
print(f"Feedback stored: {stats.total_feedback}")
print(f"Model improvement potential: +{simulate_model_improvement(stats.total_feedback) - 0.85:.1%}")
print("="*60)

print("\n✅ HITL simulation completed successfully!")